This script implements Gradient Descent, a fundamental optimization algorithm used to minimize a function by iteratively updating parameters.
It includes:
- Visualizing a simple quadratic function
- Implementing gradient descent for function minimization
- Applying gradient descent to a linear regression problem
- Using mini-batch gradient descent for improved efficiency

In [1]:
import numpy as np

from plotly.graph_objects import *
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import random

import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split


Function to minimize
--
$$z = (x-1)^2 + (y-2)^2$$

In [2]:
# Define the function to be minimized: z = (x - 1)^2 + (y - 2)^2
x = np.linspace(-150, 150)
y = np.linspace(-150, 150)


xx,yy=np.meshgrid(x,y)
zz = (xx-1)**2+(yy-2)**2

In [3]:
# Visualizing the function as a 3D surface plot
fig = Figure(data = Surface(x = xx, y = yy, z= zz))
fig.show()

In [4]:
# Contour plot for better visualization of function shape
fig = Figure(data=Contour(x=x,y=y,z=zz,colorscale='viridis', contours_coloring='lines'),
            layout = Layout(width = 600, height = 600))
fig.show()

Gradient Descent Algorithm
---

$$z = (x-1)^2 + (y-2)^2$$

$$\frac{\partial z}{\partial x} = 2\cdot(x-1)$$

$$\frac{\partial z}{\partial y} = 2\cdot(y-2)$$


$$x_{n+1} = x_n-\alpha\cdot \frac{\partial z}{\partial x}$$

$$y_{n+1} = y_n-\alpha\cdot \frac{\partial z}{\partial y}$$

In [5]:
def gradient_descent(learning_rate=0.1,
                     iterations = 100, #epochs
                     initial_x = 0 , initial_y = 0):
  """Performs gradient descent to minimize the function z = (x - 1)^2 + (y - 2)^2"""
  x, y = initial_x, initial_y #initialisation
  x_list, y_list = [x], [y]
  z_list = [(x-1)**2+(y-2)**2]

  for i in range(iterations):
    gradient_x = 2*(x-1) # Partial derivative w.r.t x
    gradient_y = 2*(y-2) # Partial derivative w.r.t y
    x = x - learning_rate*gradient_x
    y = y - learning_rate*gradient_y
    x_list.append(x)
    y_list.append(y)
    z_list.append((x-1)**2 + (y-2)**2)

  return x_list, y_list, z_list

In [6]:
# Run gradient descent
x_gd, y_gd, z_gd = gradient_descent()

In [7]:
# Visualizing the descent path
x, y = np.linspace(-2,2), np.linspace(-2,2)
xx, yy = np.meshgrid(x,y)
zz = (xx-1)**2+(yy-2)**2

fig = Figure(data = [Surface(x = xx,
                             y = yy,
                             z = zz,
                             opacity = 0.4),
                     Scatter3d(x = x_gd,
                              y = y_gd,
                               z = z_gd)],
             layout = Layout(width = 600,
                             height = 600))
fig.show()

Linear Regression with Gradient Descent
---

$$\hat y = w_1 + w_2\cdot x $$

Minimize $(y_i-\hat{y_i})^2$ for every $i$

$$L = \frac{1}{n}\sum_{i=1}^n(w_1 + w_2\cdot x_i - y_i)^2$$


$$\frac{\partial L}{\partial w_1} = \frac{2}{n}\sum_{i=1}^n(w_1 + w_2\cdot x_i - y_i)$$

$$\frac{\partial L}{\partial w_2} = \frac{2}{n}\sum_{i=1}^n x_i(w_1 + w_2\cdot x_i - y_i)$$

In [8]:
# Generate noisy data
x = np.linspace(0,100,100)
y = 2*x+1 + np.random.randn(100)*20

px.scatter(x=x,y=y)

In [9]:
def gradient_descent(x, y , learning_rate = 0.0005,
                     iterations = 1000,
                     initial_w1 = 4,
                     initial_w2 = 4):
  w1 = initial_w1
  w2 = initial_w2

  w1_list, w2_list = [w1], [w2]
  n = len(x)

  for _ in range(iterations):

    gradient_w1 = (2/n)*np.sum(w1+w2*x-y)
    gradient_w2 = (2/n)*np.sum(x*(w1+w2*x-y))

    gradient_w1 = np.clip(gradient_w1, -1000, 1000) # Avoid exploding gradients
    gradient_w2 = np.clip(gradient_w2, -1000, 1000)

    w1 = w1 - learning_rate*gradient_w1
    w2 = w2 - learning_rate*gradient_w2

    w1_list.append(w1)
    w2_list.append(w2)

  return w1_list, w2_list

In [10]:
# Run gradient descent for linear regression
w1_list, w2_list = gradient_descent(x,y)

In [11]:
# Visualize the fitted line
fig = px.scatter(x=x, y = y)
fig.add_trace(Scatter(x = x,
                       y = w1_list[-1]+x*w2_list[-1]))
fig.show()

# Mini-Batch Gradient Descent

In [12]:
def mini_batch_gradient_descent(x, y , learning_rate = 0.0005,
                     iterations = 1000,
                     initial_w1 = 4,
                     initial_w2 = 4,
                              batch_size = 10):
  w1 = initial_w1
  w2 = initial_w2

  w1_list, w2_list = [w1], [w2]
  n = len(x)

  for _ in range(iterations):
    indicies = np.random.permutation(n)
    for i in range(0,n, batch_size):
        batch_indicies = indicies[i:i+batch_size]
        x_batch = x[batch_indicies]
        y_batch = y[batch_indicies]

        gradient_w1 = (2/n)*np.sum(w1+w2*x_batch-y_batch)
        gradient_w2 = (2/n)*np.sum(x_batch*(w1+w2*x_batch-y_batch))

        gradient_w1 = np.clip(gradient_w1, -1000, 1000)
        gradient_w2 = np.clip(gradient_w2, -1000, 1000)

        w1 = w1 - learning_rate*gradient_w1
        w2 = w2 - learning_rate*gradient_w2

    w1_list.append(w1)
    w2_list.append(w2)

  return w1_list, w2_list

In [13]:
w1_list, w2_list = mini_batch_gradient_descent(x,y)

In [14]:
fig = px.scatter(x=x, y = y)
fig.add_trace(Scatter(x = x,
                       y = w1_list[-1]+x*w2_list[-1]))
fig.show()

Using Keras
--

In [15]:
# Reshape x for Keras (it expects a 2D array for features)
x = x.reshape(-1, 1)

In [16]:
# Split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [17]:
# Build the model
inputs = keras.Input(shape=(1,))
outputs = layers.Dense(1)(inputs)
model = keras.Model(inputs, outputs)

In [18]:
# Compile the model
model.compile(
    optimizer = 'adam',
    loss='mse', # Mean Squared Error for regression
    metrics=['mae'] # Mean Absolute Error for regression
)

In [19]:
# Train the model
history = model.fit(
    x_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - loss: 9505.8203 - mae: 82.9906 - val_loss: 11986.0840 - val_mae: 91.2570
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 9484.4023 - mae: 82.8895 - val_loss: 11959.5742 - val_mae: 91.1519
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 9460.6758 - mae: 82.7857 - val_loss: 11933.2598 - val_mae: 91.0475
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 9439.0273 - mae: 82.6853 - val_loss: 11906.7920 - val_mae: 90.9423
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 9417.3018 - mae: 82.5842 - val_loss: 11880.2627 - val_mae: 90.8368
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 9395.4678 - mae: 82.4843 - val_loss: 11853.7461 - val_mae: 90.7312
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 9373.2520 - mae: 82.3834 - val_loss: 11827.2988 - val_mae: 90.6258
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 9351.5488 - mae: 82.2819 - val_loss: 11800.8818 - val_mae: 90.5203


In [20]:
# Evaluate the model
test_loss, test_mae = model.evaluate(x_test, y_test)
print(f"Test Loss: {test_loss}")
print(f"Test MAE: {test_mae}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 7394.5498 - mae: 74.2126
Test Loss: 7394.5498046875
Test MAE: 74.212646484375


In [21]:
# Get the weights and bias
weights = model.get_weights()
w = weights[0][0][0]
b = weights[1][0]
print(f"Learned parameter w: {w}")
print(f"Learned parameter b: {b}")
print(f"True parameters: w=2, b=1")

Learned parameter w: 0.5821762681007385
Learned parameter b: 0.19387438893318176
True parameters: w=2, b=1


In [22]:
# Make predictions
y_pred = model.predict(x).flatten()

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


In [23]:
# Create a dataframe for easier plotting
df = pd.DataFrame({
    'x': x.flatten(),
    'y': y,
    'y_pred': y_pred
})
df = df.sort_values('x')  # Sort for line plotting

In [24]:
# Create subplots: one for data and prediction, one for training history
fig = make_subplots(rows=2, cols=1,
                    subplot_titles=('Linear Regression with Keras',
                                   'Training and Validation Loss'),
                    vertical_spacing=0.15,
                    specs=[[{"type": "scatter"}],
                           [{"type": "scatter"}]])
# Add scatter plot for original data
fig.add_trace(
    go.Scatter(
        x=df['x'],
        y=df['y'],
        mode='markers',
        name='Data',
        marker=dict(color='blue', size=8)
    ),
    row=1, col=1
)
# Add line plot for model prediction
fig.add_trace(
    go.Scatter(
        x=df['x'],
        y=df['y_pred'],
        mode='lines',
        name='Model Prediction',
        line=dict(color='red', width=2)
    ),
    row=1, col=1
)

# Add training loss plot
fig.add_trace(
    go.Scatter(
        x=list(range(len(history.history['loss']))),
        y=history.history['loss'],
        mode='lines',
        name='Training Loss',
        line=dict(color='blue', width=2)
    ),
    row=2, col=1
)
# Add validation loss plot
fig.add_trace(
    go.Scatter(
        x=list(range(len(history.history['val_loss']))),
        y=history.history['val_loss'],
        mode='lines',
        name='Validation Loss',
        line=dict(color='green', width=2)
    ),
    row=2, col=1
)

# Update layout
fig.update_layout(
    height=800,
    width=900,
    title_text="Linear Regression Analysis",
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Update axes
fig.update_xaxes(title_text="x", row=1, col=1)
fig.update_yaxes(title_text="y", row=1, col=1)
fig.update_xaxes(title_text="Epoch", row=2, col=1)
fig.update_yaxes(title_text="Loss", row=2, col=1)